# 🐝 Heady Brain — GPU Runtime

**Chat directly with HeadyBrain. Keys load automatically — just run the cells.**

1. Run Cell 1 (loads keys)
2. Run Cell 2 (chat engine with **persistent conversation history**)
3. Run Cell 3 to talk!

### Commands
- `chat("msg")` — continues conversation
- `new_chat()` — fresh conversation
- `save_chat("name")` / `load_chat("name")` — persist across restarts
- `show_history()` — view conversation so far

In [ ]:
# ═══ Cell 1: Load Keys from Heady Memory ═══import base64 as _b, osdef _d(s): return _b.b64decode(s).decode()_vault = {    'gemini': [        (_d('QUl6YVN5RE9MNVlKX042Q0xlVk5ReXRCeURENm5aMEktS3I0WVJr'), 'HC-Heady'),        (_d('QUl6YVN5QllSWEtTckdDVXBfWUxhbGo5R0NRNnRtbjlNZENpVXNn'), 'HC-GCloud'),        (_d('QUl6YVN5Q3hkTlowQUpFVVoxUi1iQWljaDJIa28tRHpSb0w4RjVv'), 'HM-2'),        (_d('QUl6YVN5RENZTXJjYkVONTF1d0lteXR2RUtPeUVFbXVaZGc4TFBr'), 'HM-3'),        (_d('QUl6YVN5QkxUdTBoOVEwOUNyMDVfM19aal8zeWVudDVjTzNpYUhF'), 'HM-4'),        (_d('QUl6YVN5REIxU2MyUHpzUld4TldvaGNJQ0dIQXFRS001WXZSY1hv'), 'HM-Default'),    ],    'openai': [        (_d('c2stcHJvai1mb29fcDRGa2JCWklDTENRcFc4T3VmMlRLTlNZVlBrTnd5Q3lKUlBDQ29xVmFITWVLT3NKSF9Ha0h5bHhKR1JzZW85VHQySVo3QlQzQmxia0ZKT2ZXNThjTTh6LUFDMXY5dUt4YjdkelpoMjNLXy1XM0lmb0U5NkxCME44NXJlVEp2V1Mxa0llOHFxM2ZkTzlsdEVVUHJWOUNWVUE='), 'Seat1'),        (_d('c2stc3ZjYWNjdC0xNzB6NFhLYnI4dE0xTDBRbkpsMEMtS0lOeldaemFLM0E3RVpIaDJMRjNHczJqUUplZEpuallJRUFHb3A2OUwxbGY1WGwyajcyS1QzQmxia0ZKdTNHRnFZUVZxdDVuc2ZxdmlFNm1xUXVyUmlYM19DbXk1TVYxRjlFdUVzSU1odHNZaTVyNmZ2YURnWGRYWWN4VGZ0cUZWZXctd0E='), 'SvcAcct'),    ],    'anthropic': [        (_d('c2stYW50LWFwaTAzLXN4cnZjRkVmdzBZMkJVWnVySVNidmhaS0N0OWg3bTNyRDZXa3hUaDJKekR4U2N0MlFENjZTZTEybjRjbkIxUVM1Nk1IYWNQWFgwVERXUG8wM0ppSU1RLWRRYkJid0FB'), 'Claude-API03'),    ],    'huggingface': [        (_d('aGZfdW1IdXlLV1pVU2tXaG54U3RaQVFORUdtSFNXenBlU0dLaw=='), 'HF-Seat1'),        (_d('aGZfT3lwU0JwSVlNaERrR3BEVkZoYVZJSUxPRmVnUlpWV3hPdA=='), 'HF-Seat2'),    ],    'perplexity': [],}def _get(name):    try:        from google.colab import userdata        v = userdata.get(name)        if v: return v    except: pass    return os.environ.get(name)for name, provider in [('GEMINI_API_KEY','gemini'),('OPENAI_API_KEY','openai'),('ANTHROPIC_API_KEY','anthropic'),('HF_TOKEN','huggingface'),('PERPLEXITY_API_KEY','perplexity')]:    v = _get(name)    if v and not any(k==v for k,_ in _vault.get(provider,[])): _vault[provider].append((v, f'env:{name}'))KEYS = _vaulttotal = sum(len(v) for v in KEYS.values())print(f'🐝 Heady Memory: {total} keys loaded automatically')for p, ks in KEYS.items():    if ks: print(f'  ✅ {p}: {len(ks)} key(s)')    else:  print(f'  ⬜ {p}: none')

In [ ]:
# ═══ Cell 2: Liquid Chat Engine with Determinism + Persistent History ═══
import json, urllib.request, urllib.error, time, concurrent.futures, os, uuid, difflib

SYSTEM = """You are HeadyBrain, the sovereign AI reasoning engine of the Heady ecosystem.
3 accounts (HeadySystems, HeadyConnection, HeadyMe), 5 domains, HuggingFace Spaces, Cloud Run.
Be helpful, precise, and actionable."""

# ─── Persistent Conversation History ───────────────────────────
conversation_history = []
session_id = str(uuid.uuid4())[:8]
MAX_HISTORY = 50
SESSIONS_DIR = '/content/heady_sessions'

def _trim():
    global conversation_history
    if len(conversation_history) > MAX_HISTORY:
        conversation_history = conversation_history[-MAX_HISTORY:]

def new_chat():
    global conversation_history, session_id
    conversation_history = []
    session_id = str(uuid.uuid4())[:8]
    print(f'New conversation (session: {session_id})')

def show_history():
    if not conversation_history:
        print('No history yet.')
        return
    for i, msg in enumerate(conversation_history):
        role = 'You' if msg['role'] == 'user' else 'Heady'
        print(f'  {i+1}. [{role}]: {msg["content"][:200]}')

def save_chat(name='auto'):
    os.makedirs(SESSIONS_DIR, exist_ok=True)
    path = os.path.join(SESSIONS_DIR, f'{name}.json')
    with open(path, 'w') as f:
        json.dump({'session_id': session_id, 'history': conversation_history, 'saved_at': time.strftime('%Y-%m-%dT%H:%M:%S')}, f, indent=2)
    print(f'Saved {len(conversation_history)} messages to {path}')

def load_chat(name='auto'):
    global conversation_history, session_id
    path = os.path.join(SESSIONS_DIR, f'{name}.json')
    if not os.path.exists(path):
        print(f'No saved chat: {path}')
        return
    with open(path) as f:
        data = json.load(f)
    conversation_history = data.get('history', [])
    session_id = data.get('session_id', name)
    print(f'Loaded {len(conversation_history)} messages (session: {session_id})')


# ─── Safe HTTP ─────────────────────────────────────────────────
def _http(url, data, headers, timeout=120):
    req = urllib.request.Request(url, data=data, headers=headers)
    try:
        return json.loads(urllib.request.urlopen(req, timeout=timeout).read())
    except urllib.error.HTTPError as e:
        body = ''
        try: body = e.read().decode()[:300]
        except: pass
        raise RuntimeError(f'HTTP {e.code}: {body}')


# ─── Provider Calls ────────────────────────────────────────────
GEMINI_MODELS = ['gemini-2.5-pro', 'gemini-2.5-flash', 'gemini-3.1-pro-preview']

def _call_gemini(key, message):
    contents = []
    for msg in conversation_history:
        role = 'model' if msg['role'] == 'assistant' else 'user'
        contents.append({'role': role, 'parts': [{'text': msg['content']}]})
    contents.append({'role': 'user', 'parts': [{'text': message}]})
    body = json.dumps({
        'systemInstruction': {'parts': [{'text': SYSTEM}]},
        'contents': contents,
        'generationConfig': {'maxOutputTokens': 8192}
    }).encode()
    last_err = None
    for model in GEMINI_MODELS:
        url = f'https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={key}'
        try:
            d = _http(url, body, {'Content-Type': 'application/json'})
            if 'candidates' in d and d['candidates']:
                parts = d['candidates'][0].get('content', {}).get('parts', [])
                if parts and parts[0].get('text', '').strip():
                    return parts[0]['text']
            last_err = f'{model}: empty response'
        except Exception as e:
            last_err = f'{model}: {e}'
    raise RuntimeError(last_err or 'all gemini models failed')

def _call_openai(key, message):
    messages = [{'role': 'system', 'content': SYSTEM}]
    messages.extend(conversation_history)
    messages.append({'role': 'user', 'content': message})
    d = _http('https://api.openai.com/v1/chat/completions',
        json.dumps({'model': 'gpt-4o', 'messages': messages, 'max_tokens': 8192}).encode(),
        {'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
    if d.get('choices'):
        return d['choices'][0]['message']['content']
    raise RuntimeError(f'no choices: {json.dumps(d)[:200]}')

def _call_anthropic(key, message):
    messages = list(conversation_history)
    messages.append({'role': 'user', 'content': message})
    d = _http('https://api.anthropic.com/v1/messages',
        json.dumps({'model': 'claude-sonnet-4-20250514', 'max_tokens': 8192, 'system': SYSTEM, 'messages': messages}).encode(),
        {'Content-Type': 'application/json', 'x-api-key': key, 'anthropic-version': '2023-06-01'})
    if d.get('content'):
        return d['content'][0]['text']
    raise RuntimeError(f'no content: {json.dumps(d)[:200]}')

def _call_perplexity(key, message):
    messages = [{'role': 'system', 'content': SYSTEM}]
    messages.extend(conversation_history)
    messages.append({'role': 'user', 'content': message})
    d = _http('https://api.perplexity.ai/chat/completions',
        json.dumps({'model': 'sonar-pro', 'messages': messages, 'max_tokens': 4096}).encode(),
        {'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
    if d.get('choices'):
        return d['choices'][0]['message']['content']
    raise RuntimeError(f'no choices: {json.dumps(d)[:200]}')

def _call_huggingface(key, message):
    messages = list(conversation_history)
    messages.append({'role': 'user', 'content': message})
    d = _http('https://router.huggingface.co/novita/v3/openai/chat/completions',
        json.dumps({'model': 'deepseek/deepseek-r1-0528', 'messages': messages, 'max_tokens': 4096, 'stream': False}).encode(),
        {'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})
    if d.get('choices'):
        return d['choices'][0]['message']['content']
    raise RuntimeError(f'no choices: {json.dumps(d)[:200]}')

PROVIDERS = {
    'gemini': _call_gemini,
    'openai': _call_openai,
    'anthropic': _call_anthropic,
    'huggingface': _call_huggingface,
    'perplexity': _call_perplexity,
}
PROVIDER_ORDER = ['gemini', 'openai', 'anthropic', 'huggingface', 'perplexity']


# ─── Determinism Engine ───────────────────────────────────────
def _similarity(a, b):
    """Semantic similarity between two responses (0.0 - 1.0)."""
    a_words = set(a.lower().split())
    b_words = set(b.lower().split())
    if not a_words or not b_words:
        return 0.0
    jaccard = len(a_words & b_words) / len(a_words | b_words)
    seq = difflib.SequenceMatcher(None, a[:2000], b[:2000]).ratio()
    return (jaccard + seq) / 2

DETERMINISM_THRESHOLD = 0.4  # minimum similarity for agreement

def _deterministic_call(message):
    """Intelligent determinism: race ALL available providers in parallel.
    As responses arrive, check for agreement. Stop as soon as minimum
    proof of determinism is achieved (2+ providers agree).
    Returns the consensus response with a determinism score."""
    available = []
    for p in PROVIDER_ORDER:
        keys = KEYS.get(p, [])
        fn = PROVIDERS.get(p)
        if keys and fn:
            available.append((p, keys[0][0], keys[0][1], fn))

    if not available:
        return {'ok': False, 'response': 'No providers available'}

    if len(available) == 1:
        p, key, label, fn = available[0]
        try:
            s = time.time()
            r = fn(key, message)
            return {'ok': True, 'response': r, 'provider': p, 'label': label,
                    'ms': int((time.time()-s)*1000), 'determinism': 'single-source',
                    'score': 0.0, 'agreeing': 1, 'total': 1}
        except Exception as e:
            return {'ok': False, 'response': f'{p}: {e}'}

    # Race all providers in parallel
    results = []  # (provider, label, response, ms)
    errors = []
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=len(available)) as pool:
        futs = {}
        for p, key, label, fn in available:
            fut = pool.submit(fn, key, message)
            futs[fut] = (p, label, time.time())

        # Process results as they arrive — check for agreement progressively
        for fut in concurrent.futures.as_completed(futs, timeout=120):
            p, label, t0 = futs[fut]
            try:
                r = fut.result()
                if r and r.strip():
                    ms = int((time.time() - t0) * 1000)
                    results.append((p, label, r, ms))
                    print(f'  [{p}/{label}] responded ({ms}ms, {len(r)} chars)')

                    # Check: do we have deterministic proof yet?
                    if len(results) >= 2:
                        # Find best agreement pair
                        best_sim = 0
                        best_pair = None
                        for i in range(len(results)):
                            for j in range(i+1, len(results)):
                                sim = _similarity(results[i][2], results[j][2])
                                if sim > best_sim:
                                    best_sim = sim
                                    best_pair = (i, j)

                        if best_sim >= DETERMINISM_THRESHOLD and best_pair:
                            # Determinism proven — use the longer of the agreeing pair
                            i, j = best_pair
                            winner = results[i] if len(results[i][2]) >= len(results[j][2]) else results[j]
                            agreeing_names = f'{results[i][0]}+{results[j][0]}'
                            total_ms = int((time.time() - start) * 1000)
                            print(f'  DETERMINISTIC ({agreeing_names}, similarity: {best_sim:.0%}, {total_ms}ms)')

                            # Cancel remaining futures
                            for f in futs:
                                f.cancel()

                            return {
                                'ok': True,
                                'response': winner[2],
                                'provider': winner[0],
                                'label': winner[1],
                                'ms': total_ms,
                                'determinism': 'proven',
                                'score': best_sim,
                                'agreeing': 2,
                                'total': len(available),
                                'proof': agreeing_names,
                                'all_results': [(p, l, len(r)) for p, l, r, _ in results],
                            }
            except Exception as e:
                errors.append(f'{p}/{label}: {str(e)[:80]}')
                print(f'  [{p}/{label}] failed: {str(e)[:80]}')

    total_ms = int((time.time() - start) * 1000)

    if not results:
        return {'ok': False, 'response': 'All providers failed:\n' + '\n'.join(f'  - {e}' for e in errors)}

    if len(results) == 1:
        r = results[0]
        return {'ok': True, 'response': r[2], 'provider': r[0], 'label': r[1],
                'ms': total_ms, 'determinism': 'single-source', 'score': 0.0,
                'agreeing': 1, 'total': len(available)}

    # Multiple results but none hit threshold — pick longest, report divergence
    best_sim = 0
    for i in range(len(results)):
        for j in range(i+1, len(results)):
            sim = _similarity(results[i][2], results[j][2])
            if sim > best_sim:
                best_sim = sim

    winner = max(results, key=lambda x: len(x[2]))
    return {
        'ok': True,
        'response': winner[2],
        'provider': winner[0],
        'label': winner[1],
        'ms': total_ms,
        'determinism': 'divergent',
        'score': best_sim,
        'agreeing': len(results),
        'total': len(available),
        'all_results': [(p, l, len(r)) for p, l, r, _ in results],
    }


# ─── Liquid Router ─────────────────────────────────────────────
def _liquid_call(message, preferred=None):
    """Try preferred provider, then cascade. Never crashes."""
    errors = []
    order = list(PROVIDER_ORDER)
    if preferred and preferred in order:
        order.remove(preferred)
        order.insert(0, preferred)
    for provider in order:
        keys = KEYS.get(provider, [])
        fn = PROVIDERS.get(provider)
        if not keys or not fn:
            continue
        for key, label in keys:
            try:
                s = time.time()
                r = fn(key, message)
                if r and r.strip():
                    return {'ok': True, 'response': r, 'provider': provider, 'label': label, 'ms': int((time.time()-s)*1000)}
            except Exception as e:
                errors.append(f'{provider}/{label}: {str(e)[:100]}')
    return {'ok': False, 'response': 'All providers failed:\n' + '\n'.join(f'  - {e}' for e in errors)}


# ─── Main Chat Function ───────────────────────────────────────
def chat(message, provider='auto'):
    """Liquid chat with optional deterministic mode.

    Modes:
      chat("msg")                 auto-route, liquid failover
      chat("msg", "gemini")       prefer Gemini, cascade on fail
      chat("msg", "deterministic") race all, prove determinism
      chat("msg", "all")          deep research, all providers
    """
    if provider == 'all':
        return deep_research(message)

    if provider == 'deterministic':
        print('Racing all available providers for determinism proof...\n')
        result = _deterministic_call(message)
    else:
        preferred = None if provider == 'auto' else provider
        result = _liquid_call(message, preferred)

    if result['ok']:
        conversation_history.append({'role': 'user', 'content': message})
        conversation_history.append({'role': 'assistant', 'content': result['response']})
        _trim()
        turns = len(conversation_history) // 2
        det = result.get('determinism', '')
        score = result.get('score', '')
        det_info = f' | {det} {score:.0%}' if det else ''
        print(f'\n[{result["provider"]}/{result["label"]}] ({result["ms"]}ms, {turns} turns{det_info})\n')
        print(result['response'])
        return result['response']
    else:
        print(result['response'])
        return None


def deep_research(message):
    """Query ALL providers simultaneously with full history."""
    print('Deep Research: querying all providers...\n')
    tasks = [(p, ks[0][0], ks[0][1]) for p, ks in KEYS.items() if ks and p in PROVIDERS]
    results, start = [], time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:
        futs = {pool.submit(PROVIDERS[p], k, message): (p, l) for p, k, l in tasks}
        for f in concurrent.futures.as_completed(futs):
            p, l = futs[f]
            try:
                r = f.result(timeout=120)
                if r and r.strip():
                    print(f'  OK {p}/{l}: {len(r)} chars')
                    results.append((p, l, r))
            except Exception as e:
                print(f'  FAIL {p}/{l}: {str(e)[:80]}')
    conversation_history.append({'role': 'user', 'content': message})
    if results:
        best = max(results, key=lambda x: len(x[2]))
        conversation_history.append({'role': 'assistant', 'content': best[2]})
    _trim()
    ms = int((time.time()-start)*1000)
    print(f'\n{len(results)}/{len(tasks)} responded ({ms}ms)\n')
    for p, l, r in results:
        print(f'-- {p.upper()}/{l} ({len(r)} chars) --')
        print(r[:3000])
        if len(r) > 3000:
            print(f'... [{len(r)-3000} more]')
        print()
    return results

print(f'Chat ready (session: {session_id})')
print('  chat("msg")                auto-route, liquid failover')
print('  chat("msg", "gemini")      prefer Gemini')
print('  chat("msg", "deterministic")  race all, prove determinism')
print('  chat("msg", "all")         deep research, all providers')
print('  new_chat() / show_history() / save_chat() / load_chat()')


In [ ]:
chat("What should we prioritize to get Heady to enterprise grade?")

In [ ]:
# ═══ Cell 4: 🔬 DEEP RESEARCH ═══
deep_research("Enterprise best practices for multi-provider AI with autonomous operations")

In [ ]:
# ═══ Cell 5: Interactive Chat ═══
print('HeadyBrain — type and talk! History is maintained.')
print('Prefix: @openai @claude @gemini @hf @all @det')
print('Commands: /new /save [name] /load [name] /history /quit')
print()
while True:
    try:
        msg = input('You: ')
        if msg.lower() in ('quit', 'exit', 'q', '/quit'):
            break
        if not msg.strip():
            continue
        if msg.strip() == '/new':
            new_chat()
            continue
        if msg.strip() == '/history':
            show_history()
            continue
        if msg.strip().startswith('/save'):
            parts = msg.strip().split(' ', 1)
            save_chat(parts[1] if len(parts) > 1 else 'auto')
            continue
        if msg.strip().startswith('/load'):
            parts = msg.strip().split(' ', 1)
            load_chat(parts[1] if len(parts) > 1 else 'auto')
            continue
        p = 'auto'
        for pfx, prov in [('@det ', 'deterministic'), ('@openai ', 'openai'), ('@claude ', 'anthropic'), ('@gemini ', 'gemini'), ('@hf ', 'huggingface'), ('@all ', 'all')]:
            if msg.startswith(pfx):
                p, msg = prov, msg[len(pfx):]
                break
        print()
        chat(msg, p)
        print()
    except KeyboardInterrupt:
        print('Done.')
        break
